# Stapel und Warteschlange

## Agenda

- Überblick

1. Stapel
     - ... zur Klammerpaarung
     - ... zur Auswertung von Postfix-Ausdrücken
     - ... zur Verfolgung der Ausführung und *Backtracking*
2. Warteschlangen
    - ... zur Verfolgung der Ausführung und *Backtracking*
    - ... für faire Planung (auch „Round-Robin“-Planung genannt)
    - ... zur Verteilung von Arbeit
3. Laufzeitanalyse

## Übersicht

Während der List-ADT unglaublich nützlich ist, haben beide von uns untersuchten Implementierungsarten (array-basiert und verkettet) Operationen, die in $O(N)$ Zeit laufen, was ihnen ein unvorhersehbares Laufzeitverhalten verleiht.

Indem wir jedoch die List-API weiter einschränken — insbesondere durch *die Zugriffsstellen auf entweder den Anfang oder das Ende der zugrundeliegenden Daten zu isolieren* — können wir Datenstrukturen schaffen, deren Operationen durchweg in $O(1)$ laufen und die dennoch sehr nützlich sind.

## 1. Stapel

Der **Stack** ist ein abstrakter Datentyp (ADT), der nur den Zugriff auf ein „Ende“ der Datensammlung erlaubt. Wir können nur Elemente am hinteren Ende (auch „Top“ genannt) eines Stacks anhängen („pushen“), und nur das zuletzt hinzugefügte Element kann entfernt („gepoppt“) werden. Das zuletzt auf einen Stack gelegte Element ist daher das erste, das wieder entfernt wird, weshalb wir Stacks als Last-in-First-out-(LIFO)-Strukturen bezeichnen.

![](images/stack_operations.png)

### Array-basierter Stack

In [ ]:
# array-backed implementation

class Stack:
    def __init__(self):
        self.data = []
        
    def push(self, val):
        self.data.append(val)

    def pop(self):
        assert not self.empty()
        ret = self.data[-1]
        del self.data[-1]
        return ret
    
    def peek(self):
        assert not self.empty()
        return self.data[-1]

    def empty(self):
        return len(self.data) == 0

    def __bool__(self):
        return not self.empty()

In [ ]:
s = Stack()
for x in range(10):
    s.push(x)
    print(s)

In [ ]:
s.peek()

In [ ]:
while s:
    print(s.pop())

### Einfach verketteter Stapel

In [ ]:
# linked implementation

class Stack:
    class Node:
        def __init__(self, val, next=None):
            self.val = val
            self.next  = next
    
    def __init__(self):
        self.top = None

    def push(self, val):
        self.top = Stack.Node(val, next=self.top)
        
    def pop(self):
        assert not self.empty()
        ret = self.top.val
        self.top = self.top.next
        return ret

    def peek(self):
        assert not self.empty()
        return self.top.val
        
    def empty(self):
        return self.top is None
    
    def __bool__(self):
        return not self.empty()

In [ ]:
s = Stack()
for x in range(10):
    s.push(x)

In [ ]:
s.peek()

In [ ]:
while s:
    print(s.pop())

### ... um passende Klammern zu finden

Stacks werden von Parsern verwendet, um zu entscheiden, ob Ausdrücke, die gepaarte Trennzeichen verwenden (z. B. `()`, `[]`, `<>`, `<tag></tag>`), *wohlgeformt* sind.

z. B. sind alle Klammern in `'(1 + 2 * (3 - 4 / 5 + 6) - (7 + 8))'` korrekt aufeinander abgestimmt?

In [ ]:
def check_parens(expr):
    s = Stack()
    for c in expr:
        if c == '(':
            s.push(c)
        elif c == ')':
            if s.empty():
                return False
            elif s.pop() != '(':
                return False
    return s.empty()

In [ ]:
check_parens('()')

In [ ]:
check_parens('((()))')

In [ ]:
check_parens('()(()()(()))')

In [ ]:
check_parens('(')

In [ ]:
check_parens('())')

In [ ]:
check_parens('(1 + 2 * (3 - 4 / 5 + 6) - (7 + 8))')

### ... for number conversion

In computers numbers a typically not stored in decimal format but in binary format (actually as computers only understand 0 and 1 everything is stored in binary format). 

E.g. the number $233_{10}$ is the same as $11101001_{2}$ in binary format.

So there has to be a method to convert from decimal to binary format. One simple approach is the *divide by 2* algorithm.

**Divide by 2 Algorithm**

The idea of the algorithm is very simple. We continually divide a given number by 2 and keep track of the remainder. Then we store the remainders in a stack and those will be representation in binary format.

Simple Example:

$8_{10}$ has the binary representation $1000_{2}$. Our algorithm calculates:

$8 : 2 = 4$ remainder $0$

$4 : 2 = 2$ remainder $0$

$2 : 2 = 1$ remainder $0$

$1 : 2 = 0$ remainder $1$



In [ ]:
def divideBy2(decNumber):
    remstack = Stack()

    while decNumber > 0:
        rem = decNumber % 2
        
        remstack.push(rem)
        decNumber = decNumber // 2
        

    binString = ""
    while not remstack.empty():
        binString = binString + str(remstack.pop())

    return binString


In [ ]:
print(divideBy2(8))
print(divideBy2(233))

### ... zur Verfolgung der Ausführung und Rückverfolgung

In [ ]:
maze_str = """######   
              I    #   
              # ## #   
              # ####   
              #    O   
              ######"""

def parse_maze(maze_str):
    '''Parses a string representing a maze into a 2D array.'''
    grid = []
    for line in maze_str.split('\n'):
        grid.append(['# IO'.index(c) for c in line.strip()])
    return grid

def print_maze(grid):
    '''Takes a 2D array maze representation and pretty-prints it.
       The contents of the 2D maze are in the range 0-5, which are interpreted as:
    
        0: a wall
        1: an unvisited (i.e., not previously traversed) path
        2: the maze entrance
        3: the maze exit
        4: a discovered but unvisited path
        5: a visited path
    '''
    for r in grid:
        print(''.join('# IO!+'[c] for c in r))

In [ ]:
parse_maze(maze_str)

In [ ]:
print_maze(parse_maze(maze_str))

In [ ]:
maze = parse_maze(maze_str)
maze[1][0] = maze[1][1] = 5
maze[1][2] = maze[2][1] = 4
print_maze(maze)

In [ ]:
class Move:
    '''Represents a move in the maze between orthogonally adjacent locations
      `frm` and `to`, which are both (row,col) tuples.'''
    def __init__(self, frm, to):
        self.frm = frm
        self.to  = to
        
    def __repr__(self):
        return f'({self.frm[0]},{self.frm[1]}) -> ({self.to[0]},{self.to[1]})'

def moves(maze, loc):
    '''Returns all possible moves within a maze from the provide location.'''
    moves = [Move(loc, (loc[0]+d[0], loc[1]+d[1]))
            for d in ((-1, 0), (1, 0), (0, -1), (0, 1))
            if loc[0]+d[0] in range(len(maze)) and 
               loc[1]+d[1] in range(len(maze[0])) and
               maze[loc[0]+d[0]][loc[1]+d[1]] in (1, 2, 3)]
    return moves

In [ ]:
maze = parse_maze(maze_str)
print_maze(maze)

In [ ]:
moves(maze, (1, 0))

In [ ]:
moves(maze, (1, 1))

In [ ]:
maze[1][0] = 5
moves(maze, (1, 1))

In [ ]:
from time import sleep
from IPython.display import clear_output

def mark(maze, loc):
    '''Marks a loc in the maze as having been discovered'''
    if maze[loc[0]][loc[1]] != 3:
        maze[loc[0]][loc[1]] = 4

def visit(maze, loc):
    '''Marks a loc in the maze as having been visited'''
    maze[loc[0]][loc[1]] = 5    
    
def display(maze):
    '''Prints out the maze after clearing the cell -- useful for animation.'''
    clear_output(True)
    print_maze(maze)
    sleep(0.5)

In [ ]:
def solve_maze(maze, entry):
    '''Searches for the exit in a maze starting from the given entry point.
    
       The algorithm works as follows:
       
       1. Visit the entry point and save all possible moves from that location.
       2. Remove and consider one of the saved moves. If it is the exit, we are done, 
          otherwise visit the destination and save all possible moves from there.
       3. If we run out of saved moves, we can't find an exit.
       
       When we save a move, we also mark it as "discovered" in the maze.
       
       The data structure used to save moves plays a critical role in how maze
       exploration proceeds! 
    '''
    for m in moves(maze, entry):
        save_move(m)
    visit(maze, entry)
    while not out_of_moves():
        move = next_move()
        if maze[move.to[0]][move.to[1]] == 3:
            break
        display(maze)
        visit(maze, move.to)
        for m in moves(maze, move.to):
            mark(maze, m.to)
            save_move(m)
    display(maze)

In [ ]:
move_stack = Stack()

def save_move(move):
    move_stack.push(move)

def next_move():
    return move_stack.pop()

def out_of_moves():
    return move_stack.empty()

![](images/stack-maze-traversal.jpg)

In [ ]:
maze_str = """######   
              I    #   
              # ## #   
              # ####   
              #    O   
              ######"""
solve_maze(parse_maze(maze_str), (1, 0))

In [ ]:
maze_str = """#################
              I #       #     #
              # ##### # # # # #
              #     # # # # # #
              # ### ### # # ###
              #   #       #   O
              #################"""

solve_maze(parse_maze(maze_str), (1, 0))

In [ ]:
maze_str = """#################
              I               #
              # # # # # # # # #
              # # # # # # # # #
              # ###############
              #               O
              #################"""

solve_maze(parse_maze(maze_str), (1, 0))

Da der Stapel eine Last-In-First-Out-Datenstruktur ist, folgt er intuitiv den letzten Schritten auf dem zuletzt entdeckten Pfad, bis er entweder den Ausgang oder eine Sackgasse erreicht. Dann nimmt er den zuvor entdeckten Pfad wieder auf. Diese Art der Erkundung bezeichnen wir als *Tiefensuche*. (depth-first traversal).

## 2. Warteschlangen

Die **Queue** ist ein ADT, der es nur erlaubt, Elemente am Ende anzufügen ("enqueue") und Elemente vom Anfang zu entfernen ("dequeue"). Das älteste Element, das sich noch in der Queue befindet, ist daher das nächste, das entfernt wird, weshalb wir eine Queue als eine First-In-First-Out-(FIFO)-Struktur bezeichnen. Es ist hilfreich, sich eine Queue als Modell für eine Warteschlange an einer typischen Supermarktkasse vorzustellen (der erste Kunde, der ankommt, ist auch der erste, der bedient wird).

![](images/queue_visualization.png)

### Array-basierte Warteschlange

In [ ]:
# array-backed implementation

class Queue:
    def __init__(self):
        self.data = []
        self.head = -1

    def enqueue(self, val): # O(1)
        self.data.append(val)
        
    def dequeue(self): # O(1), but very space inefficient!
        assert not self.empty()
        self.head += 1
        ret = self.data[self.head]
        self.data[self.head] = None
        return ret

    def empty(self):
        return self.head + 1 == len(self.data)
        
    def __bool__(self):
        return not self.empty()

In [ ]:
q = Queue()
for x in range(10):
    q.enqueue(x)

In [ ]:
while q:
    print(q.dequeue())

### Einfach verkettete Warteschlange

In [ ]:
# linked implementation

class Queue:
    class Node:
        def __init__(self, val, next=None):
            self.val = val
            self.next  = next
    
    def __init__(self):
        self.head = self.tail = None

    def enqueue(self, val): # O(1)
        if self.tail:
            self.tail.next = self.tail = Queue.Node(val)
        else:
            self.head = self.tail = Queue.Node(val)
    
    def dequeue(self): # O(1)
        assert not self.empty()
        ret = self.head.val
        self.head = self.head.next
        if not self.head:
            self.tail = None
        return ret
    
    def empty(self):
        return self.head is None

    def __bool__(self):
        return not self.empty()

In [ ]:
q = Queue()
for x in range(10):
    q.enqueue(x)

In [ ]:
while q:
    print(q.dequeue())

### ... zur Verfolgung der Ausführung und Rückverfolgung

In [ ]:
move_queue = Queue()

def save_move(move):
    move_queue.enqueue(move)

def next_move():
    return move_queue.dequeue()

def out_of_moves():
    return move_queue.empty()

In [ ]:
maze_str = """######   
              I    #   
              # ## #   
              # ####   
              #    O   
              ######"""

solve_maze(parse_maze(maze_str), (1, 0))

In [ ]:
maze_str = """#################
              I #       #     #
              # ##### # # # # #
              #     # # # # # #
              # ### ### # # ###
              #   #       #   O
              #################"""

solve_maze(parse_maze(maze_str), (1, 0))

In [ ]:
maze_str = """#################
              I               #
              # # # # # # # # #
              # # # # # # # # #
              # ###############
              #               O
              #################"""

solve_maze(parse_maze(maze_str), (1, 0))

Intuitiv, da die Warteschlange eine First-In-First-Out – also eine *faire* – Datenstruktur ist, durchläuft sie alle Pfade, die noch nicht in einer Sackgasse endeten, und macht dabei jedes Mal nur einen Schritt weiter nach unten. Wir nennen diese Art der Erkundung *Breitensuche*.

Gibt es Arten von Labyrinthen, die sich besser mit dem einen oder dem anderen Ansatz lösen lassen (also Tiefensuche vs. Breitensuche)?

### ... für faire Planung (auch bekannt als "Round-Robin"-Planung)

Queues werden häufig verwendet, um Ressourcen auf faire Weise an verschiedene Einheiten zuzuweisen, die diese benötigen. Beispielsweise kann ein Betriebssystem eine Queue verwenden, um die Prozessorzeit auf verschiedene laufende Jobs auf einem Computer zu verteilen. Ein **Round-Robin-Scheduler** erlaubt es jedem Job, für eine feste *Zeitscheibe* auf dem Prozessor zu laufen; wenn er in dieser Zeit nicht fertig wird, tritt er wieder am Ende der Queue ein:

![](images/job_queue_visualization.png)

Hier implementieren wir einen "Round-Robin"-Scheduler, der es verschiedenen Aufgaben erlaubt, für kleine, feste Zeitabschnitte zu laufen, bis sie abgeschlossen sind:

In [ ]:
from random import randint

# create a bunch of jobs with random lengths
job_queue = Queue()
for i in range(5):
    job_queue.enqueue((f'Job {i}', randint(1, 5)))

# manually print out the jobs
n = job_queue.head
while n:
    print(n.val)
    n = n.next

In [ ]:
from time import sleep

# scheduler loop
while job_queue:
    job, time_left = job_queue.dequeue()  # grab job at front of queue
    print(f'[\x1b[31mRUNNING\x1b[0m] {job}')
    sleep(1)  # run it for 1 second
    time_left -= 1
    
    # requeue if necessary
    if time_left > 0:
        print(f'[\x1b[33mREQUEUE\x1b[0m] {job} with time remaining = {time_left}')
        job_queue.enqueue((job, time_left))
    else:
        print(f'[\x1b[32mCOMPLETED\x1b[0m] {job}')

### ... zur Arbeitsverteilung

Queues werden auch häufig als eine Art Förderband verwendet, von dem ein Pool homogener Arbeiter Aufgaben entnimmt.

![](images/queue_as_conveyor_belt.png)

Hier implementieren wir dieses „Work-Queue“-Muster und verwenden es, um Arbeitselemente an einen Pool von gleichzeitig ausgeführten Threads zu übermitteln:

In [ ]:
from threading import Thread
from queue import Queue
from time import sleep
from random import random


class Worker(Thread):
    def __init__(self, wid, queue):
        super().__init__()
        self.wid= wid
        self.queue = queue
        
    def run(self):
        print(f'Worker {self.wid} starting up')
        while True:
            work = self.queue.get()    # retrieve a work item from the queue
            if work == 'Stop':
                print(f'Worker {self.wid} stopping.')
                break
            else:
                print(f'Worker {self.wid} processing {work}')
                sleep(random())        # pretend to do some work (with random duration)
                self.queue.task_done() # indicate that we've finished the work item
                

# create a work queue
work_queue = Queue()

# create a bunch of workers that monitor the queue for work items
for i in range(5):
    w = Worker(i, work_queue)
    w.start()

In [ ]:
# add a bunch of work items to the queue
for i in range(50):
    work_queue.put(i)
    
# wait for all work items to be processed
work_queue.join()

In [ ]:
# order all workers to terminate
for i in range(5):
    work_queue.put('Stop')

## 3. Laufzeitanalyse

Stack- & Queue-Implementierungen:

- Einfügen (push und enqueue) = $O(1)$
- Löschen (pop und dequeue) = $O(1)$